In [1]:
from decouple import AutoConfig
config = AutoConfig(search_path='./../.env')

In [2]:
import os
import openai

openai.api_key = config('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = openai.api_key

### Defining the LLM 

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gpt-4o-mini'
llm = ChatOpenAI(
    model=model_name,
    temperature=0.0,
    max_tokens=1024
)

llm

### Defining the search tools

In [4]:
from langchain_community.tools.arxiv.tool import ArxivQueryRun
from langchain_community.tools.tavily_search import TavilySearchResults

arxiv_search = ArxivQueryRun()
tavily_tool = TavilySearchResults(max_results=5)

tools = [arxiv_search, tavily_tool]


### Defining the Graph state

In [5]:
from typing import TypedDict, Annotated, List, Union
from IPython.display import Image, display

In [6]:
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    input: str
    agent_outcome: Union[AnyMessage, None]
    chat_history: Annotated[list[AnyMessage], add_messages]

### Initialising the workflow(graph)

In [7]:
from langgraph.graph import StateGraph
workflow = StateGraph(AgentState)

### Defining the agent (node) 

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableConfig

def research_agent(data, config:RunnableConfig):
    print("----research node----")
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful AI research assistant chatbot,"
                " Use the appropriate search tools to progress towards finding the relevant results."
                " Once you have the relevant search results, summarise them to answer the user query."
                "\nYou have access to the following search tools: {tool_names}."
            ),
            (
                "human",
                "\nUser Query: {input}"
            ),
            
            MessagesPlaceholder(variable_name="chat_history"),
        ]
    )
    prompt = prompt.partial(tool_names=", ".join([tool.name for tool in tools]))
    agent = prompt | llm.bind_tools(tools)
    result = agent.invoke(data, config=config)
    return {'agent_outcome': [result],
            'chat_history': [result]}

In [ ]:
from langgraph.graph import END, StateGraph
workflow = StateGraph(AgentState)

workflow.add_node("research", research_agent)
workflow.set_entry_point("research")

In [ ]:
import json
from langchain_core.messages import ToolMessage

class BasicToolNode:
    def __init__(self, tools: list) -> None:
        self.tools_by_name = {tool.name: tool for tool in tools}

    def __call__(self, inputs: dict):
        print("----tool calling----")
        message = inputs["agent_outcome"][-1]

        outputs = []
        for tool_call in message.tool_calls:
            print(f"---- Calling {tool_call['name']} with args: {tool_call['args']} ----")
            tool_result = self.tools_by_name[tool_call["name"]].invoke(
                tool_call["args"]
            )
            outputs.append(
                ToolMessage(
                    content=json.dumps(tool_result),
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )

        return {
                "agent_outcome": outputs,
                "chat_history": outputs
            }

tool_node = BasicToolNode(tools=tools)
workflow.add_node("tools", tool_node)

In [11]:
def route_tools(
    state: AgentState,
):
    """
    Use in the conditional_edge to route to the ToolNode if the last message
    has tool calls. Otherwise, route to the end.
    """
    print("----router----")
    if isinstance(state, list):
        ai_message = state[-1]
    elif agent_outcome := state.get("agent_outcome", []):
        ai_message = agent_outcome[-1]
    else:
        raise ValueError(f"No messages found in input state to tool_edge: {state}")

    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        return "tools"
    return END

In [ ]:
workflow.add_conditional_edges(
    "research",
    route_tools,
    {"tools": "tools", END: END}
)

In [ ]:
workflow.add_edge("tools", "research")

In [14]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

In [ ]:
app = workflow.compile(checkpointer=memory)
try:
    display(Image(app.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
inputs = {
    "input": "What are Small Language Models?",
}

config = {
    "configurable": {
        "thread_id": "1",
    }
}

inputs["chat_history"] = HumanMessage(inputs["input"])

for step in app.stream(inputs, config=config, stream_mode="values"):
    try:
        for msg in step['agent_outcome']:
            msg.pretty_print()
    except:
        HumanMessage(inputs["input"]).pretty_print()

### Streaming Tokens

Various node level updates (**events**) can be streamed using [`.astream_events`](https://langchain-ai.github.io/langgraph/concepts/streaming/#streaming-llm-tokens-and-events-astream_events).

**Event** is a `dict` object. The important keys are:
1. `event`: type of event that is being emitted. 
2. `name`: name of the vent.
3. `data`: data associated with the event.
4. `metadata`: contains the `langgraph_node`. 



In [ ]:
from langchain_core.messages import HumanMessage

inputs = {
    "input": "What are Small Language Models?",
}

config = {
    "configurable": {
        "thread_id": "2",
    }
}

inputs["chat_history"] = HumanMessage(inputs["input"])

state = AgentState(**inputs)
async for event in app.astream_events(input=state, config=config, version='v2'):
    print(event)

#### What all is stremed for an event?

In [ ]:
event.keys()

##### Digging deep in each of the keys.

In [ ]:
async for event in app.astream_events(input=state, config=config, version='v2'):
    print(event['event'])

In [ ]:
async for event in app.astream_events(input=state, config=config, version='v2'):
    print(event['data'])

This is agent state associated with the event.

In [ ]:
async for event in app.astream_events(input=state, config=config, version='v2'):
    print(event['name'])

In [ ]:
async for event in app.astream_events(input=state, config=config, version='v2'):
    print(event['metadata'])

#### Filtered event info

In [ ]:

config = {
    "configurable": {
        "thread_id": "3",
    }
}

async for event in app.astream_events(input=state, config=config, version='v2'):
    print(f"Node: {event['metadata'].get('langgraph_node')}. Type: {event['event']}. Name: {event['name']}")

**Tokens** will be generated during `on_chain_stream` event. And it will be generated by the researcher and tools node.

In [ ]:

inputs = {
    "input": "How are LLMs different from Small Language Models?",
}
inputs["chat_history"] = HumanMessage(inputs["input"])
state = AgentState(**inputs)

config = {
    "configurable": {
        "thread_id": "4",
    }
}
nodes_to_stream = ["research", "tools"]
async for event in app.astream_events(input=state, config=config, version='v2'):
    if event['event'] == "on_chat_model_stream" and event['metadata'].get('langgraph_node', '') in nodes_to_stream:
        print(event["data"])

In the first `researcher` execution, it is identifying the  tool call, function arrguement is getting streamed. During the second `researcher` execution, it is generating tokens, which is getting streamed.

In [ ]:

inputs = {
    "input": "How are LLMs different from Small Language Models?",
}
inputs["chat_history"] = HumanMessage(inputs["input"])
state = AgentState(**inputs)

config = {
    "configurable": {
        "thread_id": "5",
    }
}
nodes_to_stream = ["research", "tools"]
async for event in app.astream_events(input=state, config=config, version='v2'):
    if event['event'] == "on_chat_model_stream" and event['metadata'].get('langgraph_node', '') in nodes_to_stream:
        print(event["data"]['chunk'].content, end='|', flush=True)